# MADAD — 08 Final XGBoost Test Evaluation

## Locked P10 / P50 / P90 XGBoost

This is the **final one-time Test evaluation**.

- Model family selected using Validation: **XGBoost Quantile**
- Hyperparameters and boosting rounds are locked from Validation.
- No Test-based tuning or model selection.
- Final models are retrained on **Train + Validation**.
- Test remains isolated until prediction/evaluation.


In [1]:
# ============================================================
# 1) Mount Drive and set paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_PATH = Path(
    '/content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset'
)

DATA_PATH = BASE_PATH / 'Madad_Integrated_Data.csv'

OUTPUT_PATH = (
    BASE_PATH
    / 'forecasting_outputs_clean'
    / '08_final_xgboost_test'
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print('Data path:', DATA_PATH)
print('Exists:', DATA_PATH.exists())
print('Save folder:', OUTPUT_PATH)

# ============================================================
# 2) Install and import packages
# ============================================================
!pip -q install "xgboost>=2.0" joblib

import warnings
warnings.filterwarnings('ignore')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_pinball_loss,
    r2_score
)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

RANDOM_STATE = 42
EPS = 1e-6


Mounted at /content/drive
Data path: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/Madad_Integrated_Data.csv
Exists: True
Save folder: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/forecasting_outputs_clean/08_final_xgboost_test


In [2]:
# ============================================================
# 3) Load integrated Madad data
# ============================================================
df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

df['date_parsed'] = pd.to_datetime(
    df['date_parsed'],
    errors='coerce'
)

group_cols = ['hf_pk', 'productID']

df_model = (
    df
    .dropna(
        subset=[
            'date_parsed',
            'hf_pk',
            'productID',
            'consumption'
        ]
    )
    .copy()
)

df_model = (
    df_model
    .sort_values(group_cols + ['date_parsed'])
    .reset_index(drop=True)
)

print('Integrated rows:', f'{len(df_model):,}')
print('Facilities:', df_model['hf_pk'].nunique())
print('Products:', df_model['productID'].nunique())
print(
    'Date range:',
    df_model['date_parsed'].min(),
    'to',
    df_model['date_parsed'].max()
)

print('\nConsumption summary:')
print(df_model['consumption'].describe())


Integrated rows: 457,225
Facilities: 1091
Products: 36
Date range: 2019-10-01 00:00:00 to 2023-11-01 00:00:00

Consumption summary:
count    457225.000000
mean        161.034016
std         378.274544
min           0.000000
25%          10.000000
50%          50.000000
75%         173.000000
max       48000.000000
Name: consumption, dtype: float64


In [3]:
# ============================================================
# 4) Calendar-safe helper functions
# ============================================================
def calendar_month_gap(current_date, previous_date):
    return (
        (current_date.dt.year - previous_date.dt.year) * 12
        + (current_date.dt.month - previous_date.dt.month)
    )


def add_calendar_safe_lags(
    data,
    group_cols,
    date_col,
    value_cols,
    lags
):
    data = data.copy()

    for col in value_cols:
        if col not in data.columns:
            continue

        grouped = data.groupby(
            group_cols,
            sort=False
        )

        for lag in lags:
            previous_value = grouped[col].shift(lag)
            previous_date = grouped[date_col].shift(lag)

            gap = calendar_month_gap(
                data[date_col],
                previous_date
            )

            data[f'{col}_lag_{lag}'] = (
                previous_value.where(gap == lag)
            )

    return data


def add_calendar_lagged_aggregate_features(
    base_df,
    group_keys,
    source_col,
    prefix,
    lags=range(1, 7),
    agg='mean'
):
    # Monthly group aggregate is built using only that month's rows.
    monthly = (
        base_df
        .groupby(
            group_keys + ['date_parsed'],
            as_index=False
        )[source_col]
        .agg(agg)
        .rename(columns={source_col: '_aggregate_value'})
    )

    result = base_df

    for lag in lags:
        lagged = monthly.copy()

        lagged['date_parsed'] = (
            lagged['date_parsed']
            + pd.offsets.MonthBegin(lag)
        )

        lagged = lagged.rename(
            columns={
                '_aggregate_value':
                f'{prefix}_lag_{lag}'
            }
        )

        result = result.merge(
            lagged[
                group_keys
                + ['date_parsed', f'{prefix}_lag_{lag}']
            ],
            on=group_keys + ['date_parsed'],
            how='left'
        )

    return result


In [4]:
# ============================================================
# 5) Local temporal features BEFORE target construction
# ============================================================
lag_source_cols = [
    c for c in [
        'consumption',
        'received',
        'openBalance',
        'closeBalance',
        'stockout'
    ]
    if c in df_model.columns
]

df_model = add_calendar_safe_lags(
    data=df_model,
    group_cols=group_cols,
    date_col='date_parsed',
    value_cols=lag_source_cols,
    lags=range(1, 13)
)

for col in [
    'consumption',
    'received',
    'closeBalance'
]:
    if col not in df_model.columns:
        continue

    for window, min_history in [
        (3, 2),
        (6, 3),
        (12, 6)
    ]:
        lag_cols = [
            f'{col}_lag_{lag}'
            for lag in range(1, window + 1)
            if f'{col}_lag_{lag}' in df_model.columns
        ]

        history_count = (
            df_model[lag_cols]
            .notna()
            .sum(axis=1)
        )

        df_model[
            f'{col}_history_count_{window}'
        ] = history_count

        df_model[
            f'{col}_roll_mean_{window}'
        ] = (
            df_model[lag_cols]
            .mean(axis=1, skipna=True)
            .where(history_count >= min_history)
        )

        df_model[
            f'{col}_roll_std_{window}'
        ] = (
            df_model[lag_cols]
            .std(axis=1, skipna=True)
            .where(history_count >= min_history)
        )

for window, min_history in [
    (3, 2),
    (6, 3),
    (12, 6)
]:
    lag_cols = [
        f'stockout_lag_{lag}'
        for lag in range(1, window + 1)
        if f'stockout_lag_{lag}' in df_model.columns
    ]

    history_count = (
        df_model[lag_cols]
        .notna()
        .sum(axis=1)
    )

    df_model[f'stockout_rate_past_{window}'] = (
        df_model[lag_cols]
        .mean(axis=1, skipna=True)
        .where(history_count >= min_history)
    )

df_model['year'] = df_model['date_parsed'].dt.year
df_model['month'] = df_model['date_parsed'].dt.month
df_model['quarter_num'] = df_model['date_parsed'].dt.quarter

df_model['month_sin'] = np.sin(
    2 * np.pi * df_model['month'] / 12
)

df_model['month_cos'] = np.cos(
    2 * np.pi * df_model['month'] / 12
)

if {
    'consumption_roll_mean_3',
    'consumption_roll_mean_6'
}.issubset(df_model.columns):
    df_model['demand_trend_3_vs_6'] = (
        df_model['consumption_roll_mean_3']
        - df_model['consumption_roll_mean_6']
    )

if {
    'consumption_roll_mean_3',
    'consumption_roll_mean_12'
}.issubset(df_model.columns):
    df_model['demand_trend_3_vs_12'] = (
        df_model['consumption_roll_mean_3']
        - df_model['consumption_roll_mean_12']
    )

if 'consumption_roll_mean_12' in df_model.columns:
    df_model['current_consumption_vs_12m'] = (
        df_model['consumption']
        / (
            df_model['consumption_roll_mean_12'].abs()
            + EPS
        )
    )

if {
    'closeBalance',
    'consumption_roll_mean_3'
}.issubset(df_model.columns):
    df_model['inventory_runway_3m'] = (
        df_model['closeBalance']
        / (
            df_model['consumption_roll_mean_3'].abs()
            + EPS
        )
    )

if {
    'received',
    'consumption'
}.issubset(df_model.columns):
    df_model['received_to_consumption'] = (
        df_model['received']
        / (
            df_model['consumption'].abs()
            + EPS
        )
    )

df_model['facility_product_observation_count_prior'] = (
    df_model
    .groupby(group_cols)
    .cumcount()
)

print('Local temporal features complete.')


Local temporal features complete.


In [5]:
# ============================================================
# 6) NEW V2 CONTEXT:
# Lagged product-level and district-product demand signals
# ============================================================
# These are all shifted forward by explicit calendar months.
# Therefore, a current row only receives aggregate demand
# information from prior calendar months.

df_model = add_calendar_lagged_aggregate_features(
    base_df=df_model,
    group_keys=['productID'],
    source_col='consumption',
    prefix='product_mean_consumption',
    lags=range(1, 7),
    agg='mean'
)

if 'district' in df_model.columns:
    df_model = add_calendar_lagged_aggregate_features(
        base_df=df_model,
        group_keys=['district', 'productID'],
        source_col='consumption',
        prefix='district_product_mean_consumption',
        lags=range(1, 7),
        agg='mean'
    )

for prefix in [
    'product_mean_consumption',
    'district_product_mean_consumption'
]:
    lag3 = [
        f'{prefix}_lag_{lag}'
        for lag in range(1, 4)
        if f'{prefix}_lag_{lag}' in df_model.columns
    ]

    lag6 = [
        f'{prefix}_lag_{lag}'
        for lag in range(1, 7)
        if f'{prefix}_lag_{lag}' in df_model.columns
    ]

    if lag3:
        df_model[f'{prefix}_roll3'] = (
            df_model[lag3]
            .mean(axis=1, skipna=True)
        )

    if lag6:
        df_model[f'{prefix}_roll6'] = (
            df_model[lag6]
            .mean(axis=1, skipna=True)
        )

    if (
        f'{prefix}_roll3' in df_model.columns
        and f'{prefix}_roll6' in df_model.columns
    ):
        df_model[f'{prefix}_trend_3_vs_6'] = (
            df_model[f'{prefix}_roll3']
            - df_model[f'{prefix}_roll6']
        )

print('Contextual aggregate features complete.')


Contextual aggregate features complete.


In [6]:
# ============================================================
# EXPERIMENT 2 — Demand-dynamics features
# ============================================================
def safe_ratio(num, den):
    return num / (np.abs(den) + EPS)

if 'consumption_lag_1' in df_model.columns:
    df_model['consumption_growth_1m'] = df_model['consumption'] - df_model['consumption_lag_1']
    df_model['consumption_growth_rate_1m'] = safe_ratio(df_model['consumption'] - df_model['consumption_lag_1'], df_model['consumption_lag_1'])

if {'consumption_roll_mean_3','consumption_roll_mean_6'}.issubset(df_model.columns):
    df_model['recent_vs_6m_ratio'] = safe_ratio(df_model['consumption_roll_mean_3'], df_model['consumption_roll_mean_6'])

if {'consumption_roll_mean_3','consumption_roll_mean_12'}.issubset(df_model.columns):
    df_model['recent_vs_12m_ratio'] = safe_ratio(df_model['consumption_roll_mean_3'], df_model['consumption_roll_mean_12'])

for window in [3,6,12]:
    mean_col=f'consumption_roll_mean_{window}'
    std_col=f'consumption_roll_std_{window}'
    if {mean_col,std_col}.issubset(df_model.columns):
        df_model[f'consumption_cv_{window}m'] = safe_ratio(df_model[std_col], df_model[mean_col])
    lag_cols=[f'consumption_lag_{lag}' for lag in range(1,window+1) if f'consumption_lag_{lag}' in df_model.columns]
    if lag_cols:
        valid_count=df_model[lag_cols].notna().sum(axis=1)
        zero_count=df_model[lag_cols].eq(0).sum(axis=1)
        df_model[f'zero_consumption_rate_{window}m'] = zero_count / valid_count.replace(0,np.nan)

if 'consumption_roll_mean_6' in df_model.columns:
    df_model['demand_spike_ratio'] = safe_ratio(df_model['consumption'], df_model['consumption_roll_mean_6'])

if {'consumption_lag_1','consumption_lag_3'}.issubset(df_model.columns):
    df_model['consumption_momentum_3m'] = df_model['consumption_lag_1'] - df_model['consumption_lag_3']

print('Experiment 2 demand-dynamics features added.')

Experiment 2 demand-dynamics features added.


In [7]:
# ============================================================
# 7) Construct NEXT-CALENDAR-MONTH demand target
# ============================================================
grouped = df_model.groupby(
    group_cols,
    sort=False
)

df_model['next_date_parsed'] = (
    grouped['date_parsed']
    .shift(-1)
)

df_model['next_month_consumption_observed'] = (
    grouped['consumption']
    .shift(-1)
)

df_model['month_start'] = (
    df_model['date_parsed']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_model['next_month_start'] = (
    df_model['next_date_parsed']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_model['expected_next_month'] = (
    df_model['month_start']
    + pd.offsets.MonthBegin(1)
)

is_true_next_month = (
    df_model['next_month_start']
    == df_model['expected_next_month']
)

df_model['target_consumption_next_month'] = (
    np.where(
        is_true_next_month,
        df_model['next_month_consumption_observed'],
        np.nan
    )
)

df_model = (
    df_model
    .dropna(
        subset=['target_consumption_next_month']
    )
    .copy()
)

target_gap = calendar_month_gap(
    df_model['next_month_start'],
    df_model['month_start']
)

assert (target_gap == 1).all()

print(
    'Eligible forecasting rows:',
    f'{len(df_model):,}'
)

print(
    'Target relation:',
    'ALL rows are exactly next calendar month.'
)

print('\nTarget summary:')
print(
    df_model[
        'target_consumption_next_month'
    ].describe()
)


Eligible forecasting rows: 313,698
Target relation: ALL rows are exactly next calendar month.

Target summary:
count    313698.000000
mean        167.782345
std         400.037253
min           0.000000
25%          10.000000
50%          50.000000
75%         195.000000
max       48000.000000
Name: target_consumption_next_month, dtype: float64


In [8]:
# ============================================================
# 8) Chronological whole-month split
# ============================================================
df_model = (
    df_model
    .sort_values('date_parsed')
    .reset_index(drop=True)
)

unique_dates = np.array(
    sorted(
        df_model[
            'date_parsed'
        ].dropna().unique()
    )
)

n_dates = len(unique_dates)

train_cut = (
    max(
        1,
        int(n_dates * 0.715)
    )
    - 1
)

val_cut = (
    max(
        train_cut + 1,
        int(
            n_dates
            * (0.715 + 0.127)
        )
    )
    - 1
)

train_end_date = pd.Timestamp(
    unique_dates[train_cut]
)

val_end_date = pd.Timestamp(
    unique_dates[
        min(
            val_cut,
            n_dates - 1
        )
    ]
)

train_df = df_model[
    df_model['date_parsed']
    <= train_end_date
].copy()

val_df = df_model[
    (
        df_model['date_parsed']
        > train_end_date
    )
    & (
        df_model['date_parsed']
        <= val_end_date
    )
].copy()

test_df = df_model[
    df_model['date_parsed']
    > val_end_date
].copy()

print(
    'Train:',
    f'{len(train_df):,}',
    train_df['date_parsed'].min(),
    'to',
    train_df['date_parsed'].max()
)

print(
    'Validation:',
    f'{len(val_df):,}',
    val_df['date_parsed'].min(),
    'to',
    val_df['date_parsed'].max()
)

print(
    'Test:',
    f'{len(test_df):,}',
    test_df['date_parsed'].min(),
    'to',
    test_df['date_parsed'].max()
)

print(
    'Total:',
    f'{len(train_df) + len(val_df) + len(test_df):,}'
)

assert (
    train_df['date_parsed'].max()
    < val_df['date_parsed'].min()
)

assert (
    val_df['date_parsed'].max()
    < test_df['date_parsed'].min()
)


Train: 211,452 2019-10-01 00:00:00 to 2022-08-01 00:00:00
Validation: 39,864 2022-09-01 00:00:00 to 2023-02-01 00:00:00
Test: 62,382 2023-03-01 00:00:00 to 2023-10-01 00:00:00
Total: 313,698


In [9]:
# ============================================================
# 9) Evaluation helpers
# ============================================================
TARGET = 'target_consumption_next_month'

def regression_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    denominator = np.sum(
        np.abs(y_true)
    )

    wape = (
        np.sum(
            np.abs(
                y_true - y_pred
            )
        )
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape,
        'R2': r2
    }


def print_metrics(
    name,
    y_true,
    y_pred
):
    metrics = regression_metrics(
        y_true,
        y_pred
    )

    print(f'--- {name} ---')
    print(
        f"MAE : "
        f"{metrics['MAE']:.4f}"
    )
    print(
        f"RMSE: "
        f"{metrics['RMSE']:.4f}"
    )
    print(
        f"WAPE: "
        f"{metrics['WAPE']*100:.2f}%"
    )
    print(
        f"R²  : "
        f"{metrics['R2']:.4f}"
    )

    return metrics


In [10]:
# ============================================================
# 10) LOCK FINAL TRAINING DATA
# ============================================================
# Model selection is finished.
# Retrain on Train + Validation only.
# Test is kept completely separate.

trainval_df = (
    pd.concat([train_df, val_df], axis=0)
    .sort_values('date_parsed')
    .reset_index(drop=True)
)

print('Final Train+Validation:', f'{len(trainval_df):,}')
print(
    trainval_df['date_parsed'].min(),
    'to',
    trainval_df['date_parsed'].max()
)

print('Final untouched Test:', f'{len(test_df):,}')
print(
    test_df['date_parsed'].min(),
    'to',
    test_df['date_parsed'].max()
)

assert trainval_df['date_parsed'].max() < test_df['date_parsed'].min()


Final Train+Validation: 251,316
2019-10-01 00:00:00 to 2023-02-01 00:00:00
Final untouched Test: 62,382
2023-03-01 00:00:00 to 2023-10-01 00:00:00


In [11]:
# ============================================================
# 11) Select leakage-safe forecasting features
# ============================================================
base_numeric = [
    'consumption_growth_1m',
    'consumption_growth_rate_1m',
    'recent_vs_6m_ratio',
    'recent_vs_12m_ratio',
    'consumption_cv_3m',
    'consumption_cv_6m',
    'consumption_cv_12m',
    'zero_consumption_rate_3m',
    'zero_consumption_rate_6m',
    'zero_consumption_rate_12m',
    'demand_spike_ratio',
    'consumption_momentum_3m',

    'consumption',
    'received',
    'openBalance',
    'closeBalance',
    'normAvg',
    'normStd',
    'year',
    'month',
    'quarter_num',
    'month_sin',
    'month_cos',
    'facility_product_observation_count_prior',
    'demand_trend_3_vs_6',
    'demand_trend_3_vs_12',
    'current_consumption_vs_12m',
    'inventory_runway_3m',
    'received_to_consumption'
]

dynamic_features = [
    c for c in df_model.columns
    if any(
        c.startswith(prefix)
        for prefix in [
            'consumption_lag_',
            'received_lag_',
            'openBalance_lag_',
            'closeBalance_lag_',
            'stockout_lag_',
            'consumption_roll_',
            'received_roll_',
            'closeBalance_roll_',
            'consumption_history_count_',
            'received_history_count_',
            'closeBalance_history_count_',
            'stockout_rate_past_',
            'product_mean_consumption_',
            'district_product_mean_consumption_'
        ]
    )
]

numeric_features = [
    c
    for c in (
        base_numeric
        + dynamic_features
    )
    if c in trainval_df.columns
]

numeric_features = list(
    dict.fromkeys(
        numeric_features
    )
)

categorical_features = [
    c for c in [
        'hf_pk',
        'productID',
        'facility_type',
        'district'
    ]
    if c in trainval_df.columns
]

numeric_features = [
    c
    for c in numeric_features
    if not trainval_df[c].isna().all()
]

feature_cols = (
    numeric_features
    + categorical_features
)

print(
    'Numeric features:',
    len(numeric_features)
)

print(
    'Categorical features:',
    len(categorical_features)
)

print(
    'Total raw features:',
    len(feature_cols)
)


Numeric features: 137
Categorical features: 4
Total raw features: 141


In [12]:
# ============================================================
# 12) FINAL PREPROCESSING — fit on Train + Validation
# ============================================================
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop'
)

X_trainval = trainval_df[feature_cols]
X_test = test_df[feature_cols]

y_trainval = trainval_df[TARGET].values
y_test = test_df[TARGET].values

X_trainval_t = preprocessor.fit_transform(X_trainval)
X_test_t = preprocessor.transform(X_test)

print('Transformed Train+Validation:', X_trainval_t.shape)
print('Transformed Test:', X_test_t.shape)
print('TEST HAS NOT BEEN USED FOR FITTING.')


Transformed Train+Validation: (251316, 1283)
Transformed Test: (62382, 1283)
TEST HAS NOT BEEN USED FOR FITTING.


In [13]:
# ============================================================
# 13) LOCKED FINAL CONFIGURATIONS
# ============================================================
# These settings come from Validation only.
# No tuning occurs in this notebook.

P10_CONFIG = {
    'n_estimators': 600,
    'max_depth': 5,
    'learning_rate': 0.03,
    'min_child_weight': 5,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.10,
    'reg_lambda': 2.0
}

# Validation winner: candidate 3; best_iteration = 1199
P50_CONFIG = {
    'n_estimators': 1200,
    'max_depth': 6,
    'learning_rate': 0.03,
    'min_child_weight': 5,
    'subsample': 0.90,
    'colsample_bytree': 0.90,
    'reg_alpha': 0.10,
    'reg_lambda': 2.0
}

# Validation winner: candidate 5; best_iteration = 1193
P90_CONFIG = {
    'n_estimators': 1194,
    'max_depth': 6,
    'learning_rate': 0.05,
    'min_child_weight': 10,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.20,
    'reg_lambda': 3.0
}

print('Locked P10:', P10_CONFIG)
print('Locked P50:', P50_CONFIG)
print('Locked P90:', P90_CONFIG)


Locked P10: {'n_estimators': 600, 'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_alpha': 0.1, 'reg_lambda': 2.0}
Locked P50: {'n_estimators': 1200, 'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.1, 'reg_lambda': 2.0}
Locked P90: {'n_estimators': 1194, 'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 10, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_alpha': 0.2, 'reg_lambda': 3.0}


In [14]:
# ============================================================
# 14) FINAL TRAINING — Train + Validation ONLY
# ============================================================
def make_quantile_model(alpha, config):
    return XGBRegressor(
        objective='reg:quantileerror',
        quantile_alpha=alpha,
        random_state=RANDOM_STATE,
        tree_method='hist',
        device='cuda',
        n_jobs=-1,
        **config
    )

p10_model = make_quantile_model(0.10, P10_CONFIG)
p50_model = make_quantile_model(0.50, P50_CONFIG)
p90_model = make_quantile_model(0.90, P90_CONFIG)

print('Training final P10...')
p10_model.fit(X_trainval_t, y_trainval, verbose=False)

print('Training final P50...')
p50_model.fit(X_trainval_t, y_trainval, verbose=False)

print('Training final P90...')
p90_model.fit(X_trainval_t, y_trainval, verbose=False)

print('Final training complete.')


Training final P10...
Training final P50...
Training final P90...
Final training complete.


In [15]:
# ============================================================
# 15) ONE-TIME FINAL TEST PREDICTIONS
# ============================================================
p10_raw = np.clip(p10_model.predict(X_test_t), 0, None)
p50_raw = np.clip(p50_model.predict(X_test_t), 0, None)
p90_raw = np.clip(p90_model.predict(X_test_t), 0, None)

# Enforce non-crossing quantiles row-wise, same as Validation.
q = np.sort(
    np.column_stack([p10_raw, p50_raw, p90_raw]),
    axis=1
)

test_p10 = q[:, 0]
test_p50 = q[:, 1]
test_p90 = q[:, 2]

print('Test predictions generated:', len(test_p50))


Test predictions generated: 62382


In [16]:
# ============================================================
# 16) FINAL TEST METRICS
# ============================================================
test_p10_pinball = mean_pinball_loss(y_test, test_p10, alpha=0.10)
test_p50_pinball = mean_pinball_loss(y_test, test_p50, alpha=0.50)
test_p90_pinball = mean_pinball_loss(y_test, test_p90, alpha=0.90)

test_p50_metrics = regression_metrics(y_test, test_p50)

test_coverage = np.mean(
    (y_test >= test_p10)
    & (y_test <= test_p90)
)

test_interval_width = np.mean(test_p90 - test_p10)

print('=' * 72)
print('MADAD — FINAL XGBOOST TEST RESULTS')
print('=' * 72)
print('P10 Pinball:', round(test_p10_pinball, 4))
print('P50 Pinball:', round(test_p50_pinball, 4))
print('P90 Pinball:', round(test_p90_pinball, 4))
print('P50 MAE     :', round(test_p50_metrics['MAE'], 4))
print('P50 RMSE    :', round(test_p50_metrics['RMSE'], 4))
print('P50 WAPE    :', f"{test_p50_metrics['WAPE']*100:.2f}%")
print('P50 R²      :', round(test_p50_metrics['R2'], 4))
print('P10–P90 Coverage:', f'{test_coverage*100:.2f}%')
print('Mean interval width:', round(test_interval_width, 2))
print('=' * 72)


MADAD — FINAL XGBOOST TEST RESULTS
P10 Pinball: 12.5628
P50 Pinball: 38.137
P90 Pinball: 27.9193
P50 MAE     : 76.274
P50 RMSE    : 322.3729
P50 WAPE    : 53.68%
P50 R²      : 0.3107
P10–P90 Coverage: 81.51%
Mean interval width: 222.38


In [17]:
# ============================================================
# 17) VALIDATION vs FINAL TEST — reporting only
# ============================================================
# Validation numbers are fixed historical results from model selection.
# They are shown only for reporting, not for changing the model.

comparison = pd.DataFrame([
    {
        'Split': 'Validation',
        'P50_MAE': 95.0108,
        'P50_RMSE': 323.9083,
        'P50_WAPE_%': 53.14,
        'P50_R2': 0.3350,
        'P10_Pinball': 15.8208,
        'P50_Pinball': 47.5054,
        'P90_Pinball': 36.5385,
        'Coverage_%': 80.67
    },
    {
        'Split': 'Final Test',
        'P50_MAE': test_p50_metrics['MAE'],
        'P50_RMSE': test_p50_metrics['RMSE'],
        'P50_WAPE_%': test_p50_metrics['WAPE'] * 100,
        'P50_R2': test_p50_metrics['R2'],
        'P10_Pinball': test_p10_pinball,
        'P50_Pinball': test_p50_pinball,
        'P90_Pinball': test_p90_pinball,
        'Coverage_%': test_coverage * 100
    }
])

display(comparison)


,Split,P50_MAE,P50_RMSE,P50_WAPE_%,P50_R2,P10_Pinball,P50_Pinball,P90_Pinball,Coverage_%
0,Validation,95.010800,323.908300,53.140000,0.335000,15.820800,47.505400,36.538500,80.670000
1,Final Test,76.274027,322.372906,53.678648,0.310712,12.562762,38.137014,27.919334,81.507486


In [18]:
# ============================================================
# 18) SAVE FINAL TEST RESULTS, PREDICTIONS, MODELS
# ============================================================
final_summary = pd.DataFrame([{
    'Model': 'Final XGBoost Quantile',
    'Training_Data': 'Train + Validation',
    'Test_Rows': len(test_df),
    'P50_MAE': test_p50_metrics['MAE'],
    'P50_RMSE': test_p50_metrics['RMSE'],
    'P50_WAPE': test_p50_metrics['WAPE'],
    'P50_R2': test_p50_metrics['R2'],
    'P10_Pinball': test_p10_pinball,
    'P50_Pinball': test_p50_pinball,
    'P90_Pinball': test_p90_pinball,
    'Coverage_P10_P90': test_coverage,
    'Mean_Interval_Width': test_interval_width
}])

pred_df = test_df[
    ['hf_pk', 'productID', 'date_parsed', TARGET]
].copy()

for col in ['facility_type', 'district', 'closeBalance']:
    if col in test_df.columns:
        pred_df[col] = test_df[col].values

pred_df['P10'] = test_p10
pred_df['P50'] = test_p50
pred_df['P90'] = test_p90
pred_df['Absolute_Error_P50'] = np.abs(
    pred_df[TARGET] - pred_df['P50']
)

final_summary.to_csv(
    OUTPUT_PATH / 'final_xgboost_test_metrics.csv',
    index=False
)

comparison.to_csv(
    OUTPUT_PATH / 'validation_vs_final_test.csv',
    index=False
)

pred_df.to_csv(
    OUTPUT_PATH / 'final_xgboost_test_predictions.csv',
    index=False
)

joblib.dump(
    preprocessor,
    OUTPUT_PATH / 'final_preprocessor.joblib'
)

joblib.dump(
    p10_model,
    OUTPUT_PATH / 'final_xgboost_p10_model.joblib'
)

joblib.dump(
    p50_model,
    OUTPUT_PATH / 'final_xgboost_p50_model.joblib'
)

joblib.dump(
    p90_model,
    OUTPUT_PATH / 'final_xgboost_p90_model.joblib'
)

with open(
    OUTPUT_PATH / 'locked_final_configs.json',
    'w'
) as f:
    json.dump(
        {
            'P10': P10_CONFIG,
            'P50': P50_CONFIG,
            'P90': P90_CONFIG
        },
        f,
        indent=2
    )

print('Saved final outputs to:')
print(OUTPUT_PATH)
print()
display(final_summary)


Saved final outputs to:
/content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/forecasting_outputs_clean/08_final_xgboost_test



,Model,Training_Data,Test_Rows,P50_MAE,P50_RMSE,P50_WAPE,P50_R2,P10_Pinball,P50_Pinball,P90_Pinball,Coverage_P10_P90,Mean_Interval_Width
0,Final XGBoost Quantile,Train + Validation,62382,76.274027,322.372906,0.536786,0.310712,12.562762,38.137014,27.919334,0.815075,222.37706


### The final XGBoost model generalized consistently to the unseen test period, achieving an MAE of 76.27 and a P10–P90 coverage of 81.51%. Although relative forecast error remains substantial, the model provides well-calibrated uncertainty estimates suitable for downstream inventory planning.